In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import glob
import numpy as np
import os

import librosa
import soundfile as sf
import sounddevice as sd
from pydub import AudioSegment

In [3]:
root_audios = '/Users/tomasandrade/Documents/BSC/anti-spoof/dataset/ASVspoof2019/LA/ASVspoof2019_LA_train/flac'
files = sorted(glob.glob(f'{root_audios}/LA_T_1*.flac'))

output_dir = '/Users/tomasandrade/Documents/BSC/anti-spoof/dataset/ASV_preproc_v3/flac'
os.makedirs(output_dir, exist_ok=True)

In [4]:
def clip_audio(audio_path, out_folder = 'pre-process'):

    #print(f'working on file {audio_path}')

    t0, t1 = get_t0_t1(audio_path, threshold = 0.05)

    audio = AudioSegment.from_file(audio_path)
    clip = audio[int(t0*1000):int(t1*1000)]  
    clip = clip.fade_in(20).fade_out(20)  

    name = audio_path.split('/')[-1]
    file_out = f'{out_folder}/{name}'
    clip.export(file_out, format="flac")

def get_t0_t1(audio_path, threshold = 0.05):
    t, rms = get_rms(audio_path, target_sr = 16_000)

    real_threshold = threshold*rms.max()

    t0 = t[rms > real_threshold][0] 
    t1 = t[rms > real_threshold][-1] 
    return t0, t1

def get_rms(audio_path, target_sr = 16_000):

    data, samplerate = librosa.load(audio_path, sr=target_sr, mono=True)

    hop = target_sr // 50       
    rms = librosa.feature.rms(y=data, hop_length = hop)[0]
    time = np.arange(len(rms)) / 50.0
    
    return time, rms

def get_duration_flac(path):
    info = sf.info(path)
    return info.duration

In [5]:
len_in_data = sum([get_duration_flac(f) for f in files])/3600.
print(f'Total input dataset {len_in_data} hours')

Total input dataset 2.6686492881944397 hours


In [ ]:
for file in files:
    clip_audio(file, out_folder = output_dir)

In [7]:
out_files = glob.glob(f'{output_dir}/*.flac')
total_len = sum([get_duration_flac(f) for f in out_files])/3600
print(f'Total output dataset {total_len} hours')

Total output dataset 0.005116666666666667 hours
